In [ ]:
import pandas as pd
import pickle
import importlib
import sys
import os
from markov_models import MarkovModel 
from helpers import compute_info_rate, update_values_in_csv
import warnings
import re
import numpy as np
import json

languages = ["FRA"] # ["FRA", 'JPN', 'CMN', 'VIE', 'YUE', 'ENG', 'DEU']

for language in languages:

    folder = f"produced_data/{language}"
    print(f"\nLanguage: {language}")

    for processing_type in ['phonemes']: # ['sylls', 'phonemes']

        for text_type in ['words', 'sentences']:
            print(f"\nComputing ID and IR across {text_type.upper()} at the level of {processing_type.upper()}")

            n_values = [1, 2, 3, 4]  # For bigram, trigram, and 4-gram models
            markov_models = {}

            for n in n_values:
                # Load the data
                with open(f"{folder}/phonemized_{language}.json", "r", encoding="utf-8") as f:
                    data = json.load(f)

                print(f"\n🧮 Training a Markov Model with n = {n}:")

                # Create and build the Markov model
                model = MarkovModel(n)

                # Build the markov model
                model.build(data, text_type)

                # Compute the conditional entropy (information density)
                info_density = model.compute_conditional_entropy()
                print(f"Information Density: {info_density:.4f}")

                # Compute the information rate (bits per second)
                info_rates = compute_info_rate(info_density, processing_type, language)
                print(f"Information Rate: {np.mean(info_rates):.4f}")
                
                # Update the CSV file with the computed values
                #update_values_in_csv(language, info_density, n, 'ID')
                #update_values_in_csv(language, info_rates, n, 'IR')

                # Store model for later use 
                markov_models[n] = model

                # Display exactly 3 examples
                example_count = 0
                print("\nExample probabilities (p(x, y)):")

                for (prefix, suffix), p_xy in model.cond_probs.items():
                    print(f"p({prefix} -> {suffix}) = {p_xy:.4f}")
                    example_count += 1
                    if example_count == 3:
                        break
                
                # Save the model to a file
                model.save_model(language, processing_type, text_type)

    # For plotting the information rate vs n-gram order and histogram of information rates, 
    # see the respective jupyter notebook 


Language: FRA

Computing ID across WORDS at the level of PHONEMES

🧮 Training a Markov Model with n = 1:
ngram examples: [('p',), ('u',), ('ʁ',), ('l',), ('ə',)]
Information Density: 4.6896
Information Rate: 77.2793

Example probabilities (p(x, y)):
p(() -> ə‍) = 0.0001
p(() -> ɔ̃) = 0.0179
p(() -> b) = 0.0114

✅ Saved 1-gram model to 'produced_data/FRA/phonemes/'

🧮 Training a Markov Model with n = 2:
ngram examples: [('p', 'u'), ('u', 'ʁ'), ('l', 'ə'), ('ʁ', 'w'), ('w', 'a')]
Information Density: 3.4456
Information Rate: 56.7784

Example probabilities (p(x, y)):
p(('p',) -> ə‍) = 0.0001
p(('p',) -> ɔ̃) = 0.0104
p(('p',) -> b) = 0.0006

✅ Saved 2-gram model to 'produced_data/FRA/phonemes/'

🧮 Training a Markov Model with n = 3:
ngram examples: [('p', 'u', 'ʁ'), ('ʁ', 'w', 'a'), ('w', 'a', 'j'), ('a', 'j', 'o'), ('j', 'o', 'm')]
Information Density: 2.7996
Information Rate: 46.1343

Example probabilities (p(x, y)):
p(('p', 'u') -> ə‍) = 0.0004
p(('p', 'u') -> ɔ̃) = 0.0004
p(('p', 'u')